# Project Cuối Khóa: Dự Đoán Tình Trạng Vỡ Nợ Tín Dụng

## Bối cảnh

Một công ty tài chính muốn giảm thiểu rủi ro bằng cách dự đoán khả năng khách hàng sẽ không thể trả nợ (vỡ nợ) trong tương lai. Việc xác định sớm các khách hàng có rủi ro cao sẽ giúp công ty đưa ra các quyết định kinh doanh tốt hơn, chẳng hạn như điều chỉnh hạn mức tín dụng, đề xuất các gói hỗ trợ tài chính, hoặc từ chối cho vay.

Bạn, với tư cách là một nhà khoa học dữ liệu, được giao nhiệm vụ xây dựng một mô hình học máy mạnh mẽ và đáng tin cậy sử dụng LightGBM để giải quyết bài toán này. Dữ liệu được cung cấp là bộ dữ liệu `Home Credit Default Risk` nổi tiếng từ một cuộc thi trên Kaggle.

**Đây là một bài toán phân loại nhị phân (binary classification) với dữ liệu mất cân bằng (imbalanced data) điển hình.**

## Mục tiêu Project

1.  **Xây dựng một quy trình (pipeline) hoàn chỉnh:** Từ tiền xử lý dữ liệu, huấn luyện mô hình, tinh chỉnh tham số đến đánh giá và diễn giải mô hình.
2.  **Áp dụng các kỹ thuật đã học trong khóa học:**
    - Xử lý dữ liệu (missing values, categorical features).
    - Sử dụng LightGBM làm mô hình chính.
    - Thực hiện Cross-Validation để đánh giá mô hình một cách khách quan.
    - Tinh chỉnh siêu tham số (Hyperparameter Tuning) để tối ưu hóa hiệu suất.
    - Đánh giá mô hình bằng các metric phù hợp (AUC, F1-score, etc.).
    - Diễn giải kết quả mô hình (Feature Importance, SHAP) để đưa ra các kết luận có giá trị kinh doanh.
3.  **Trình bày kết quả:** Tạo một notebook rõ ràng, mạch lạc, giải thích từng bước đi và các quyết định đã đưa ra.

## Dữ liệu

Do bộ dữ liệu gốc rất lớn, chúng ta sẽ sử dụng một phiên bản đã được rút gọn và xử lý một phần để phù hợp với mục đích của project này. Bộ dữ liệu này bao gồm các thông tin nhân khẩu học, thông tin về khoản vay, lịch sử tín dụng của khách hàng.

- **Tệp dữ liệu:** `home_credit_risk_sample.csv` (sẽ được cung cấp hoặc tải về).
- **Đặc trưng mục tiêu (Target):** Cột `TARGET`. `1` có nghĩa là khách hàng gặp khó khăn trong việc thanh toán (vỡ nợ), `0` có nghĩa là các trường hợp còn lại.

## Các bước gợi ý

Đây là một lộ trình gợi ý. Bạn có thể tự do sáng tạo và thử nghiệm các phương pháp khác nhau.

### Bước 1: Khám phá và Tiền xử lý Dữ liệu (EDA & Preprocessing)

1.  **Tải dữ liệu:** Đọc file `home_credit_risk_sample.csv` vào pandas DataFrame.
2.  **Khám phá dữ liệu:**
    - Xem kích thước của dữ liệu (`.shape`).
    - Kiểm tra các kiểu dữ liệu (`.info()`).
    - Xem thống kê mô tả (`.describe()`).
    - **Quan trọng:** Kiểm tra sự mất cân bằng của biến mục tiêu `TARGET`. Tính tỷ lệ phần trăm của lớp 1 và lớp 0.
3.  **Xử lý dữ liệu bị thiếu (Missing Values):**
    - Xác định các cột có tỷ lệ thiếu cao. Bạn có thể quyết định xóa chúng hoặc sử dụng một chiến lược điền giá trị phù hợp (ví dụ: điền bằng `median` cho số, `mode` cho hạng mục).
    - LightGBM có thể tự xử lý `np.nan`, nhưng việc hiểu và xử lý chúng một cách có chủ đích vẫn là một kỹ năng tốt.
4.  **Xử lý Đặc trưng hạng mục (Categorical Features):**
    - Xác định các cột có kiểu `object`.
    - Áp dụng `LabelEncoder` hoặc `pd.get_dummies` (One-Hot Encoding). Cân nhắc ưu và nhược điểm của mỗi phương pháp. (Gợi ý: LightGBM hoạt động tốt với `LabelEncoder`).

### Bước 2: Xây dựng Mô hình Cơ sở (Baseline Model)

1.  **Chia dữ liệu:** Chia dữ liệu thành tập huấn luyện (train) và tập kiểm tra (test) bằng `train_test_split`. Nhớ sử dụng `stratify=y` để giữ nguyên tỷ lệ mất cân bằng trong cả hai tập.
2.  **Huấn luyện mô hình cơ sở:**
    - Khởi tạo một mô hình `lgb.LGBMClassifier` với các tham số mặc định hoặc một vài tham số cơ bản.
    - **Quan trọng:** Do dữ liệu mất cân bằng, hãy sử dụng tham số `scale_pos_weight` hoặc `is_unbalance=True`. `scale_pos_weight` thường được ưu tiên hơn. Giá trị của nó nên được đặt bằng `(số lượng mẫu lớp 0) / (số lượng mẫu lớp 1)`.
    - Huấn luyện mô hình trên tập train.
3.  **Đánh giá mô hình cơ sở:**
    - Dự đoán trên tập test.
    - Tính toán và in ra các chỉ số: `roc_auc_score`, `f1_score`, `precision`, `recall`, và `confusion_matrix`. AUC là metric chính cho bài toán này.

### Bước 3: Tinh chỉnh Siêu tham số (Hyperparameter Tuning)

1.  **Định nghĩa không gian tìm kiếm:** Tạo một dictionary chứa các tham số bạn muốn tinh chỉnh và khoảng giá trị của chúng (ví dụ: `num_leaves`, `learning_rate`, `max_depth`, `reg_alpha`, `reg_lambda`, `colsample_bytree`, `subsample`).
2.  **Chọn phương pháp:** Sử dụng `RandomizedSearchCV` hoặc một thư viện tối ưu hóa Bayes như `Optuna` hoặc `Hyperopt`.
3.  **Thực hiện tìm kiếm:**
    - Chạy quá trình tìm kiếm với Cross-Validation (`cv=3` hoặc `cv=5`).
    - Đảm bảo `scoring` được đặt là `'roc_auc'`.
4.  **Lấy kết quả:** In ra bộ tham số tốt nhất và điểm AUC tương ứng.

### Bước 4: Huấn luyện và Đánh giá Mô hình Cuối cùng

1.  **Huấn luyện mô hình cuối cùng:**
    - Khởi tạo một mô hình `lgb.LGBMClassifier` mới với **bộ tham số tốt nhất** bạn vừa tìm được.
    - Huấn luyện lại mô hình trên **toàn bộ tập train**.
2.  **Đánh giá trên tập test:**
    - Dự đoán trên tập test.
    - Tính toán lại các chỉ số (`roc_auc_score`, `f1_score`, etc.) và so sánh với mô hình cơ sở. Bạn có thấy sự cải thiện không?

### Bước 5: Diễn giải Mô hình

1.  **Feature Importance:**
    - Sử dụng `lgb.plot_importance` để trực quan hóa các đặc trưng quan trọng nhất theo `gain`.
    - Viết một vài nhận xét về các yếu tố hàng đầu ảnh hưởng đến rủi ro vỡ nợ.
2.  **Phân tích với SHAP:**
    - Tạo `shap.TreeExplainer` và tính toán giá trị SHAP cho tập test.
    - Vẽ biểu đồ `shap.summary_plot` và phân tích sâu hơn về tác động của một vài đặc trưng quan trọng nhất (ví dụ: `EXT_SOURCE_2`, `EXT_SOURCE_3`). Chúng ảnh hưởng đến dự đoán như thế nào?
    - (Tùy chọn nâng cao) Chọn một khách hàng có rủi ro vỡ nợ cao và một khách hàng có rủi ro thấp, sau đó sử dụng `shap.force_plot` để giải thích lý do cho từng dự đoán.

## Tiêu chí đánh giá

- **Tính đầy đủ:** Notebook có thực hiện tất cả các bước được yêu cầu không?
- **Tính chính xác:** Code có chạy đúng và cho ra kết quả hợp lý không? Các phương pháp có được áp dụng chính xác không?
- **Sự rõ ràng và mạch lạc:** Notebook có được trình bày sạch sẽ, dễ đọc không? Có các ghi chú, giải thích bằng markdown cho các bước và quyết định quan trọng không?
- **Phân tích và Diễn giải:** Phần phân tích kết quả và diễn giải mô hình có sâu sắc và mang lại giá trị không? Bạn có rút ra được kết luận gì từ mô hình không?

Chúc bạn may mắn và hoàn thành tốt project này!